# Juliet stratified cohort + Router suite 학습

Frozen split을 변경하지 않고 Train 6,000 / Dev 1,500을 Expert→CWE→leakage-group 다양성 우선으로 선택합니다. E6는 전량 보존합니다. 각 case의 후보와 5개 Expert 작업은 하나의 물리 API 요청으로 처리되며 case별로 체크포인트됩니다.

In [11]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
COHORT_CONFIG_PATH = EVAL_ROOT / 'configs' / 'cohort_15837.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
MAX_CONCURRENCY = 1000  # OpenRouter 한도에 맞춰 낮출 수 있음
TARGET_TRUTH_RECALL = 0.95
RUN_LEARNING_CURVES = False
MLP_BATCH_SIZE = 512
MLP_MAX_EPOCHS = 100
MLP_PATIENCE = 12
MLP_LEARNING_RATE = 2e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_DEVICE = 'auto'  # CUDA 가능 시 GPU, 아니면 CPU. 'cuda'로 강제 가능


In [2]:
import json, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.select_cohort import load_cohort_config, ensure_frozen_index, build_cohort_manifests
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import (resolve_models, plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, train_utility_router, train_router_learning_curves)
from model_evaluation.adapters.llm_security import expert_assignments

config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
cohort_config = load_cohort_config(COHORT_CONFIG_PATH)
models = resolve_models(ENV_FILE)
COHORT_DIR = EVAL_ROOT / 'work' / 'cohort_15837'
RUN_DIR = EVAL_ROOT / 'work' / 'router_training_stratified_7500'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_training_stratified_7500'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
try:
    import torch
    print('MLP training device:', 'cuda' if torch.cuda.is_available() else 'cpu (CUDA unavailable)')
except ImportError:
    print('PyTorch is not installed')
print('Physical model:', models[0])

Physical model: deepseek/deepseek-v4-flash-0731


## 1. Leakage-safe stratified cohort manifest

In [3]:
index_report = ensure_frozen_index(config, mapping, progress=print)
cohort_report = build_cohort_manifests(config, cohort_config, output_directory=COHORT_DIR)
print(json.dumps(index_report, ensure_ascii=False, indent=2))
print(json.dumps(cohort_report, ensure_ascii=False, indent=2))

index: 500 packages, 500 supported scenarios
index: 1000 packages, 616 supported scenarios
index: 1500 packages, 1116 supported scenarios
index: 2000 packages, 1616 supported scenarios
index: 2500 packages, 2116 supported scenarios
index: 3000 packages, 2615 supported scenarios
index: 3500 packages, 3064 supported scenarios
index: 4000 packages, 3272 supported scenarios
index: 4500 packages, 3275 supported scenarios
index: 5000 packages, 3275 supported scenarios
index: 5500 packages, 3705 supported scenarios
index: 6000 packages, 4205 supported scenarios
index: 6500 packages, 4705 supported scenarios
index: 7000 packages, 5205 supported scenarios
index: 7500 packages, 5705 supported scenarios
index: 8000 packages, 5955 supported scenarios
index: 8500 packages, 5955 supported scenarios
index: 9000 packages, 5973 supported scenarios
index: 9500 packages, 6227 supported scenarios
index: 10000 packages, 6357 supported scenarios
index: 10500 packages, 6357 supported scenarios
index: 11000 p

## 2. Train/Dev materialization과 Semantic Analyzer cache

In [5]:
manifests = {split: COHORT_DIR / f'cohort_{split}.jsonl' for split in ('train', 'dev')}
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('train', 'dev'),
    selection_manifests=manifests, progress=print,
)
candidate_reports = {}
for split in ('train', 'dev'):
    candidate_reports[split] = cache_candidates(
        RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
    )
print(json.dumps({'materialization': materialization, 'candidates': candidate_reports}, ensure_ascii=False, indent=2))

materialize train: 1/6000
materialize train: 100/6000
materialize train: 200/6000
materialize train: 300/6000
materialize train: 400/6000
materialize train: 500/6000
materialize train: 600/6000
materialize train: 700/6000
materialize train: 800/6000
materialize train: 900/6000
materialize train: 1000/6000
materialize train: 1100/6000
materialize train: 1200/6000
materialize train: 1300/6000
materialize train: 1400/6000
materialize train: 1500/6000
materialize train: 1600/6000
materialize train: 1700/6000
materialize train: 1800/6000
materialize train: 1900/6000
materialize train: 2000/6000
materialize train: 2100/6000
materialize train: 2200/6000
materialize train: 2300/6000
materialize train: 2400/6000
materialize train: 2500/6000
materialize train: 2600/6000
materialize train: 2700/6000
materialize train: 2800/6000
materialize train: 2900/6000
materialize train: 3000/6000
materialize train: 3100/6000
materialize train: 3200/6000
materialize train: 3300/6000
materialize train: 3400/60

## 3. API 호출 계획

In [6]:
plans = {}
for split in ('train', 'dev'):
    plans[split] = plan_outcome_matrix(
        cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
print(json.dumps(plans, ensure_ascii=False, indent=2))

{
  "train": {
    "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\selections\\selected_train.jsonl",
    "selected_candidate_count": 9879,
    "positive_candidate_count": 3897,
    "hard_negative_candidate_count": 5982,
    "model": "deepseek/deepseek-v4-flash-0731",
    "assignment_count": 5,
    "expected_physical_api_requests": 5982,
    "completed_physical_api_requests": 0,
    "remaining_physical_api_requests": 5982,
    "expected_logical_expert_outcomes": 49395,
    "completed_logical_expert_outcomes": 0,
    "unexpected_existing_rows": 0,
    "outcome_path": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\outcomes\\outcomes_train.jsonl"
  },
  "dev": {
    "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\selections\\selected_dev.jsonl",
    "selected_candidate_count": 2580,
    "positive_candidate_count": 1081,
    "hard_negative_candidate_count": 1499,

## 4. Batched Expert outcome 수집 (case당 API 최대 1회, 최대 1,000건 비동기 동시 처리)

In [9]:
collection_reports = {}
for split in ('train', 'dev'):
    report = collect_outcome_matrix(
        env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / f'{split}_api_ledger.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE, max_concurrency=MAX_CONCURRENCY,
    )
    collection_reports[split] = report
    print(split, json.dumps(report, ensure_ascii=False, indent=2))
    if report['status'] != 'complete':
        print('실패한 case만 남았습니다. 성공한 case는 저장되었으며 이 셀을 다시 실행하면 실패 case부터 재개합니다.')
        break

train {
  "status": "complete",
  "stop_reason": null,
  "model": "deepseek/deepseek-v4-flash-0731",
  "cases_seen": 5982,
  "completed_cases": 5982,
  "new_cases": 0,
  "failed_cases": 0,
  "max_concurrency": 1000,
  "scheduled_api_cases": 0,
  "case_level_retry_count": 0,
  "physical_requests_this_run": 0,
  "actual_cost_usd_this_run": 0.0,
  "request_contract": "one batched detection completion per case; transient failures may retry",
  "outcome_path": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\outcomes\\outcomes_train.jsonl",
  "ledger_path": "D:\\llm-security\\Model_Evaluation\\work\\router_training_stratified_7500\\ledgers\\train_api_ledger.jsonl"
}
batched outcomes: 1495/1499 completed; failed=0; request attempts this run=5
batched outcomes: 1496/1499 completed; failed=0; request attempts this run=5
batched outcomes: 1497/1499 completed; failed=0; request attempts this run=5
batched outcomes: 1498/1499 completed; failed=0; request attempts this r

## 5. GPU Multi-task MLP + 기존 Escalation Gate

LR/GBDT는 실행하지 않습니다. 각 epoch의 learning rate와 train/validation loss가 바로 출력됩니다.

In [12]:
expected_ids = [item.assignment_id for item in expert_assignments(models)]
outcome_files = {split: RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl' for split in ('train', 'dev')}
audits = {
    split: audit_outcome_matrix(
        path, expected_assignment_ids=expected_ids,
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
    ) if path.exists() else {'complete': False, 'reason': 'missing'}
    for split, path in outcome_files.items()
}
print(json.dumps(audits, ensure_ascii=False, indent=2))
if all(item['complete'] for item in audits.values()):
    training_report = train_utility_router(
        train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
        train_selection_manifest=RUN_DIR / 'selections' / 'selected_train.jsonl',
        dev_selection_manifest=RUN_DIR / 'selections' / 'selected_dev.jsonl',
        train_cohort_manifest=manifests['train'],
        artifact_path=ARTIFACT, report_path=RESULT_DIR / 'training_report.json',
        model_ids=models, backends=('multitask_mlp',),
        seed=config.seed, target_truth_recall=TARGET_TRUTH_RECALL, mlp_device=MLP_DEVICE,
        mlp_batch_size=MLP_BATCH_SIZE, mlp_max_epochs=MLP_MAX_EPOCHS,
        mlp_patience=MLP_PATIENCE, mlp_learning_rate=MLP_LEARNING_RATE,
        mlp_weight_decay=MLP_WEIGHT_DECAY, progress=print,
    )
    print(json.dumps(training_report, ensure_ascii=False, indent=2))
    if RUN_LEARNING_CURVES:
        curve_report = train_router_learning_curves(
            train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
            train_cohort_manifest=manifests['train'],
            report_path=RESULT_DIR / 'learning_curves.json', seed=config.seed,
            target_truth_recall=TARGET_TRUTH_RECALL, backends=('multitask_mlp',),
            mlp_device=MLP_DEVICE, mlp_batch_size=MLP_BATCH_SIZE,
            mlp_max_epochs=MLP_MAX_EPOCHS, mlp_patience=MLP_PATIENCE,
            mlp_learning_rate=MLP_LEARNING_RATE, mlp_weight_decay=MLP_WEIGHT_DECAY,
            progress=print,
        )
        print(json.dumps(curve_report, ensure_ascii=False, indent=2))
else:
    print('Outcome matrix가 아직 완성되지 않았습니다. 4번 셀을 다시 실행해 남은 case를 수집하세요.')

{
  "train": {
    "row_count": 49395,
    "candidate_group_count": 9879,
    "expected_assignment_count": 5,
    "duplicate_row_count": 0,
    "incomplete_candidate_group_count": 0,
    "incomplete_preview": [],
    "expected_candidate_group_count": 9879,
    "missing_candidate_group_count": 0,
    "missing_preview": [],
    "unexpected_candidate_group_count": 0,
    "complete": true
  },
  "dev": {
    "row_count": 12900,
    "candidate_group_count": 2580,
    "expected_assignment_count": 5,
    "duplicate_row_count": 0,
    "incomplete_candidate_group_count": 0,
    "incomplete_preview": [],
    "expected_candidate_group_count": 2580,
    "missing_candidate_group_count": 0,
    "missing_preview": [],
    "unexpected_candidate_group_count": 0,
    "complete": true
  }
}


KeyboardInterrupt: 